# A14 · n=500 · Bimodal
**本 Notebook 只运行这一组，不接着运行其他分布或样本量。**
正式重复次数：1000；复用第一阶段缓存：0；新生成：1000。九种二阶段方法在同一批数据上比较，全部保留。
默认12个单线程worker，进度实时显示。结果、日志和断点仅保存在当前实验文件夹。修改范围可分批跑；继续运行自动跳过已完成记录。


In [ ]:
import os,sys,json,subprocess,time
from pathlib import Path
for name in ("OMP_NUM_THREADS","OPENBLAS_NUM_THREADS","MKL_NUM_THREADS","NUMEXPR_NUM_THREADS","BLIS_NUM_THREADS"):
    os.environ[name]="1"
search=[Path.cwd(),*Path.cwd().parents,Path.home()/"Desktop"/"毕设正式论文numerical simulation"]
ROOT=next((p for p in search if (p/"代码/runner.py").is_file()),None)
if ROOT is None:raise RuntimeError("请打开本交付文件夹或把工作目录切到它内部。")
CODE=ROOT/"代码"
GROUP_CODE=ROOT/'分组实验/A14_n500_bimodal/A14实验_代码'
GROUP=GROUP_CODE.parent
print("当前实验:",GROUP.name)
print("结果目录:",GROUP)
subprocess.run([sys.executable,"-u",str(CODE/"setup_environment.py")],check=True)


## 仅设置这一组
默认 main 跑本组正式重复次数；需要先做小检查可改成 pilot（本组100次）或 smoke（本组2次）。
START/STOP 为0起点、右端不含。保留 RESUME=True；不要同时启动其他组。

In [ ]:
STAGE = "main"
WORKERS = 12
START = 0
STOP = None
RESUME = True
assert STAGE in ("smoke","pilot","main") and 1<=WORKERS<=12
cmd=[sys.executable,"-u",str(GROUP_CODE/"run_this_experiment.py"),"--"+STAGE,"--workers",str(WORKERS)]
if RESUME:cmd+=["--resume"]
if START:cmd+=["--start",str(START)]
if STOP is not None:cmd+=["--stop",str(STOP)]
print("只运行 n=500, Bimodal；阶段:",STAGE)
print("本阶段计划上限:",2 if STAGE=="smoke" else 100 if STAGE=="pilot" else 1000)
print("命令:",subprocess.list2cmdline(cmd))


## 启动本组
手动运行下一格开始。完成后停止，不启动下一组；日志和进度持续显示。中断后可续跑。

In [ ]:
import psutil
from IPython.display import display
logpath=GROUP_CODE/"logs"/(STAGE+"_"+time.strftime("%Y%m%d_%H%M%S")+".log")
env=os.environ.copy()
env.update(PYTHONUTF8="1",PYTHONIOENCODING="utf-8",PYTHONUNBUFFERED="1")
print("日志:",logpath,flush=True)
process=None
try:
    with logpath.open("w",encoding="utf-8") as logfile:
        process=subprocess.Popen(cmd,cwd=CODE,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                                 text=True,encoding="utf-8",errors="replace",bufsize=1)
        print("主进程 PID:",process.pid,flush=True)
        for line in process.stdout:
            print(line,end="",flush=True)
            logfile.write(line);logfile.flush()
        exitcode=process.wait()
        if exitcode:raise RuntimeError(f"运行退出码 {exitcode}；见 {logpath}")
except KeyboardInterrupt:
    if process and process.poll() is None:
        try:children=psutil.Process(process.pid).children(recursive=True)
        except psutil.NoSuchProcess:children=[]
        for child in children:
            try:child.terminate()
            except psutil.NoSuchProcess:pass
        process.terminate()
        _,alive=psutil.wait_procs(children,timeout=5)
        for child in alive:
            try:child.kill()
            except psutil.NoSuchProcess:pass
        process.wait(timeout=10)
        lock=ROOT/"logs/running.lock"
        if lock.exists() and lock.read_text().strip()==str(process.pid):
            lock.unlink()
    print("\n已中断。已提交记录保留；保持 RESUME=True 可续跑。")


## 查看本组结果

In [ ]:
import pandas as pd
table=GROUP/"汇总表_CSV"/STAGE/"main_results.csv"
if table.exists():display(pd.read_csv(table))
else:print("此组尚无结果。")
print("报告与图表索引:",GROUP/"结果_MD"/(STAGE+"_results.md"))
print("PDF图:",GROUP/"图"/STAGE)
progress=GROUP_CODE/"_work"/("replications" if STAGE=="main" else STAGE)/"progress.json"
if progress.exists():display(json.loads(progress.read_text(encoding="utf-8")))


## 仅在强制退出后：检查残留锁
进程还存在时保留锁，不结束其他任务。

In [ ]:
import psutil
lock=ROOT/"logs/running.lock"
if lock.exists():
    pid=int(lock.read_text().strip())
    if psutil.pid_exists(pid):print("进程仍存在，保持锁:",pid)
    else:lock.unlink();print("已清理结束进程的残留锁。")
else:print("没有运行锁。")
